 # FINARY - Side Hustle Recommendation Model



 Model ini memprediksi:

 1. `earnings_output` (Regresi - Potensi Pendapatan USD)

 2. `success_output` (Klasifikasi - Probabilitas Sukses > 80%)

In [127]:
import os
import json
import random
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = Path("freelancer_earnings_bd.csv")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


In [128]:
df = pd.read_csv(DATA_PATH)

# --- PROXY TARGET ---
exp_multiplier = df["Experience_Level"].map({"Beginner": 1.0, "Intermediate": 1.5, "Expert": 2.5}).astype("float32")
cat_multiplier = df["Job_Category"].map({
    "Web Development": 1.8, "App Development": 2.0, "Data Entry": 0.8,
    "Digital Marketing": 1.5, "Customer Support": 0.9, "Content Writing": 1.2,
    "Graphic Design": 1.3, "SEO": 1.4
}).astype("float32")

df["Earnings_Proxy"] = df["Hourly_Rate"] * (df["Job_Duration_Days"] * 2) * exp_multiplier * cat_multiplier
earn_min = float(df["Earnings_Proxy"].min())
earn_max = float(df["Earnings_Proxy"].max())
df["target_earnings"] = ((df["Earnings_Proxy"] - earn_min) / (earn_max - earn_min)).astype("float32")

df["target_success"] = ((df["Experience_Level"] == "Expert") | (df["Hourly_Rate"] < 50)).astype("float32")

# Fitur Numerik & Kategori
numerical_features = ["Hourly_Rate", "Job_Duration_Days"]
# WAJIB: Pastikan kolom platform dan project_type masuk ke dummies
df_encoded = pd.get_dummies(df, columns=["Job_Category", "Experience_Level", "Platform", "Project_Type"], dtype=float)

feature_columns = [col for col in df_encoded.columns if 
                  col.startswith("Job_Category_") or 
                  col.startswith("Experience_Level_") or 
                  col.startswith("Platform_") or 
                  col.startswith("Project_Type_") or 
                  col in numerical_features]

X = df_encoded[feature_columns].astype("float32")
y_earn = df["target_earnings"].astype("float32")
y_succ = df["target_success"].astype("float32")

# Simpan metadata untuk API
platforms = [col.replace("Platform_", "") for col in feature_columns if col.startswith("Platform_")]
project_types = [col.replace("Project_Type_", "") for col in feature_columns if col.startswith("Project_Type_")]
job_categories = [col.replace("Job_Category_", "") for col in feature_columns if col.startswith("Job_Category_")]

In [130]:
# 2. TRAIN TEST SPLIT & SCALING
X_train, X_temp, ye_train, ye_temp, ys_train, ys_temp = train_test_split(X.values, y_earn.values, y_succ.values, test_size=0.30, random_state=SEED)
X_val, X_test, ye_val, ye_test, ys_val, ys_test = train_test_split(X_temp, ye_temp, ys_temp, test_size=0.50, random_state=SEED)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # Scaler sekarang kenal SEMUA kolom (Platform & Project Type)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, ARTIFACT_DIR / "sh_scaler.joblib")
with open(ARTIFACT_DIR / "sh_feature_columns.json", "w") as f:
    json.dump(feature_columns, f)
with open(ARTIFACT_DIR / "sh_target_stats.json", "w") as f:
    json.dump({"earn_min": earn_min, "earn_max": earn_max, "categories": job_categories, "platforms": platforms, "project_types": project_types}, f)

In [131]:
# 3. CUSTOM COMPONENTS (Sesuai Ketentuan Capstone)

@tf.keras.utils.register_keras_serializable()

class CustomDenseBlock(tf.keras.layers.Layer):

    def __init__(self, units, **kwargs):

        super().__init__(**kwargs)

        self.units = units  # Wajib disimpan ke self agar bisa dipanggil get_config

        self.dense = tf.keras.layers.Dense(units)

        self.bn = tf.keras.layers.BatchNormalization()

        self.relu = tf.keras.layers.ReLU()



    def call(self, inputs):

        x = self.dense(inputs)

        x = self.bn(x)

        return self.relu(x)



    # get_config WAJIB ADA agar model tahu cara me-load parameter 'units'

    def get_config(self):

        config = super().get_config()

        config.update({"units": self.units})

        return config



class MetricsGuardCallback(tf.keras.callbacks.Callback):

    """Custom Callback untuk memastikan target proyek (Acc >= 85%, MAE <= 0.02) tercapai."""

    def __init__(self, min_acc=0.85, max_mae=0.02):

        super().__init__()

        self.min_acc = min_acc

        self.max_mae = max_mae



    def on_epoch_end(self, epoch, logs=None):

        logs = logs or {}

        acc = logs.get("val_success_output_accuracy", 0.0)

        mae = logs.get("val_earnings_output_mae", 1.0)

        if acc >= self.min_acc and mae <= self.max_mae:

            print(f"\n[INFO] Threshold Tercapai di epoch {epoch+1}! val_acc: {acc:.4f}, val_mae: {mae:.4f}. Stop training.")

            self.model.stop_training = True

In [135]:
# %% [4. MODEL ARCHITECTURE - TURBO VERSION]
inputs = tf.keras.Input(shape=(X_train_scaled.shape[1],))

# Tambah kapasitas neuron agar MAE bisa turun di bawah 0.02
x = CustomDenseBlock(128)(inputs) # Sebelumnya 64
x = CustomDenseBlock(64)(x)  # Sebelumnya 32
x = tf.keras.layers.Dropout(0.1)(x) # Tambah sedikit dropout agar tidak overfitting

earnings_output = tf.keras.layers.Dense(1, activation="linear", name="earnings_output")(x)
success_output = tf.keras.layers.Dense(1, activation="sigmoid", name="success_output")(x)

model = tf.keras.Model(inputs=inputs, outputs=[earnings_output, success_output])

# Learning rate diperkecil sedikit (0.001) agar pencarian MAE lebih presisi
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={"earnings_output": "mse", "success_output": "binary_crossentropy"},
    metrics={
        "earnings_output": [tf.keras.metrics.MeanAbsoluteError(name="mae")],
        "success_output": [tf.keras.metrics.BinaryAccuracy(name="accuracy")]
    }
)

In [ ]:
# 5. TRAINING

callbacks = [MetricsGuardCallback(min_acc=0.85, max_mae=0.02)]



history = model.fit(

    X_train_scaled,

    {"earnings_output": ye_train, "success_output": ys_train},

    validation_data=(X_val_scaled, {"earnings_output": ye_val, "success_output": ys_val}),

    epochs=300, batch_size=64, callbacks=callbacks, verbose=1

)

Epoch 1/300


22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - earnings_output_loss: 0.6105 - earnings_output_mae: 0.6159 - loss: 1.3137 - success_output_accuracy: 0.5875 - success_output_loss: 0.6840 - val_earnings_output_loss: 0.0571 - val_earnings_output_mae: 0.1886 - val_loss: 0.6117 - val_success_output_accuracy: 0.8493 - val_success_output_loss: 0.5526
Epoch 2/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - earnings_output_loss: 0.1856 - earnings_output_mae: 0.3401 - loss: 0.6180 - success_output_accuracy: 0.8374 - success_output_loss: 0.4250 - val_earnings_output_loss: 0.0751 - val_earnings_output_mae: 0.2291 - val_loss: 0.5418 - val_success_output_accuracy: 0.8630 - val_success_output_loss: 0.4660
Epoch 3/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - earnings_output_loss: 0.1524 - earnings_output_mae: 0.3043 - loss: 0.4727 - success_output_accuracy: 0.9172 - success_output_loss: 0.3149 - val_earnings_output_loss: 0.0349 - val_earnings_output_mae: 0.1544 - val_loss: 0.4313 - val_success_output_accuracy:

In [138]:
# 6. EVALUATION & EXPORT
eval_res = model.evaluate(X_test_scaled, {"earnings_output": ye_test, "success_output": ys_test}, verbose=0, return_dict=True)
print("\n--- TEST METRICS ---")
print(f"Accuracy: {eval_res['success_output_accuracy']:.4f}")
print(f"MAE     : {eval_res['earnings_output_mae']:.4f}")




--- TEST METRICS ---
Accuracy: 0.9625
MAE     : 0.0166


In [139]:
# 7. MODEL SAVE
model.save(ARTIFACT_DIR / "sh_model.keras")
print("Model Side Hustle berhasil disimpan!")

Model Side Hustle berhasil disimpan!


In [140]:
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

ARTIFACT_DIR = Path("artifacts")

# 2. LOAD MODEL
loaded_sh_model = tf.keras.models.load_model(
    ARTIFACT_DIR / "sh_model.keras",
    custom_objects={"CustomDenseBlock": CustomDenseBlock}
)
loaded_sh_scaler = joblib.load(ARTIFACT_DIR / "sh_scaler.joblib")

with open(ARTIFACT_DIR / "sh_feature_columns.json", "r") as f:
    sh_feature_columns = json.load(f)
with open(ARTIFACT_DIR / "sh_target_stats.json", "r") as f:
    sh_target_stats = json.load(f)

job_categories = sh_target_stats["categories"]
USD_TO_IDR = 17000.0

# Bobot untuk ngitung gaji realistis
CAT_MULTIPLIER = {
    "Web Development": 1.8, "App Development": 2.0, "Data Entry": 0.8,
    "Digital Marketing": 1.5, "Customer Support": 0.9, "Content Writing": 1.2,
    "Graphic Design": 1.3, "SEO": 1.4
}

def recommend_side_hustle(payload):
    exp_level = payload.get("experience_level", "Beginner")
    avail_hours = payload.get("available_hours_per_week", 10)
    target_rate = payload.get("target_hourly_rate_usd", 10.0)
    
    total_hours_per_month = avail_hours * 4 
    duration_days = total_hours_per_month / 8.0 

    simulations = []
    for job in job_categories:
        feat_map = {col: 0.0 for col in sh_feature_columns}
        feat_map["Job_Duration_Days"] = float(duration_days)
        feat_map["Hourly_Rate"] = float(target_rate)
        
        if f"Experience_Level_{exp_level}" in feat_map:
            feat_map[f"Experience_Level_{exp_level}"] = 1.0
        if f"Job_Category_{job}" in feat_map:
            feat_map[f"Job_Category_{job}"] = 1.0
            
        simulations.append(feat_map)

    df_sim = pd.DataFrame(simulations)[sh_feature_columns]
    X_sim_scaled = loaded_sh_scaler.transform(df_sim.values).astype(np.float32)
    
    # Pakai tf.constant agar tidak muncul Warning TensorFlow
    tensor_input = tf.constant(X_sim_scaled)
    _, pred_succ_prob = loaded_sh_model(tensor_input, training=False)
    
    results = []
    for i, job in enumerate(job_categories):
        # AI menghitung Peluang Sukses
        succ_prob = float(np.clip(pred_succ_prob[i][0], 0.0, 1.0))
        
        # Matematika menghitung uang secara realistis
        earn_usd = total_hours_per_month * target_rate * CAT_MULTIPLIER.get(job, 1.0)
        earn_idr = earn_usd * USD_TO_IDR
        
        # Score = Gaji x Peluang Sukses
        score = earn_idr * succ_prob
        
        results.append({
            "job_category": job,
            "predicted_monthly_earnings_idr": round(earn_idr, 2),
            "success_probability": round(succ_prob, 4),
            "_score": score
        })
        
    results.sort(key=lambda x: x["_score"], reverse=True)
    
    top_3 = results[:3]
    for item in top_3:
        del item["_score"]

    return {
        "recommendations": top_3
    }

# --- TEST INFERENCE ---
sample_sh_payload = {
    "experience_level": "Beginner",
    "available_hours_per_week": 15,
    

}

hasil_sh = recommend_side_hustle(sample_sh_payload)
print(json.dumps(hasil_sh, indent=2))

{
  "recommendations": [
    {
      "job_category": "App Development",
      "predicted_monthly_earnings_idr": 20400000.0,
      "success_probability": 1.0
    },
    {
      "job_category": "Web Development",
      "predicted_monthly_earnings_idr": 18360000.0,
      "success_probability": 1.0
    },
    {
      "job_category": "Digital Marketing",
      "predicted_monthly_earnings_idr": 15300000.0,
      "success_probability": 1.0
    }
  ]
}
